In [12]:
import json
import time
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output


class NPZReplayViewer:
    def __init__(self, npz_path, save_path="annotations.json"):
        self.npz_path = Path(npz_path)
        self.save_path = Path(save_path)

        self.data = np.load(self.npz_path, allow_pickle=True)

        self.positions = self.data["positions"]
        self.goals = self.data["goals"]
        self.obstacles = self.data["obstacles"]
        self.segment_diffs = self.data.get("segment_diffs", None)

        self.num_steps = self.positions.shape[0]
        self.num_agents = self.positions.shape[1]
        self.num_rows, self.num_cols = self.obstacles.shape

        if self.segment_diffs is not None:
            self.num_segments = len(self.segment_diffs)
        else:
            self.num_segments = 1

        self.segment_ranges = self._infer_segment_ranges()

        self.scenario_id = self.npz_path.stem
        self.current_timestep = 0
        self.is_playing = False

        self.annotation = {
            "scenario_id": self.scenario_id,
            "npz_path": str(self.npz_path),
            "num_steps": int(self.num_steps),
            "num_segments": int(self.num_segments),
            "segment_diffs": self.segment_diffs.astype(int).tolist()
            if self.segment_diffs is not None else None,
            "worst_congestion_failure_segment_index": None,
            "clearly_clean_segment_index": None,
            "notes": {
                "worst_congestion_failure": "",
                "clearly_clean_segment": "",
            },
        }

        self.agent_colors = plt.cm.tab20(
            np.linspace(0, 1, max(self.num_agents, 1))
        )

        self._load_existing_annotation()
        self._build_widgets()

    def _infer_segment_ranges(self):
        """
        Infer segment timestep ranges.

        If your .npz eventually stores exact segment boundaries, add support here.
        For now, this evenly divides the trajectory across segment_diffs.
        """
        if self.num_segments <= 1:
            return [(0, self.num_steps - 1)]

        boundaries = np.linspace(0, self.num_steps, self.num_segments + 1).astype(int)

        ranges = []
        for i in range(self.num_segments):
            start = int(boundaries[i])
            end = int(boundaries[i + 1] - 1)
            ranges.append((start, end))

        return ranges

    def _segment_for_timestep(self, timestep):
        for i, (start, end) in enumerate(self.segment_ranges):
            if start <= timestep <= end:
                return i
        return self.num_segments - 1

    def _load_existing_annotation(self):
        if not self.save_path.exists():
            return

        try:
            with open(self.save_path, "r") as f:
                all_annotations = json.load(f)

            existing = all_annotations.get(self.scenario_id, None)

            if existing is not None:
                self.annotation.update(existing)

        except Exception:
            pass

    def _save_annotation(self):
        all_annotations = {}

        if self.save_path.exists():
            try:
                with open(self.save_path, "r") as f:
                    all_annotations = json.load(f)
            except Exception:
                all_annotations = {}

        all_annotations[self.scenario_id] = self.annotation

        with open(self.save_path, "w") as f:
            json.dump(all_annotations, f, indent=2)

    def _build_widgets(self):
        self.slider = widgets.IntSlider(
            value=0,
            min=0,
            max=self.num_steps - 1,
            step=1,
            description="Timestep:",
            continuous_update=False,
            layout=widgets.Layout(width="600px"),
        )

        self.segment_selector = widgets.IntSlider(
            value=0,
            min=0,
            max=self.num_segments - 1,
            step=1,
            description="Segment:",
            continuous_update=False,
            layout=widgets.Layout(width="600px"),
        )

        self.prev_button = widgets.Button(description="Prev")
        self.next_button = widgets.Button(description="Next")
        self.play_button = widgets.Button(description="Play")
        self.pause_button = widgets.Button(description="Pause")

        self.play_delay = widgets.FloatSlider(
            value=0.25,
            min=0.05,
            max=2.0,
            step=0.05,
            description="Delay:",
            continuous_update=False,
            layout=widgets.Layout(width="400px"),
        )

        self.mark_worst_button = widgets.Button(
            description="Mark Worst Congestion Failure"
        )
        self.mark_clean_button = widgets.Button(
            description="Mark Clearly Clean Segment"
        )
        self.clear_button = widgets.Button(description="Clear Selected Segment Mark")

        self.notes = widgets.Text(
            value="",
            placeholder="Optional note for selected mark",
            description="Note:",
            layout=widgets.Layout(width="600px"),
        )

        self.slider.observe(self._on_slider_change, names="value")
        self.segment_selector.observe(self._on_segment_change, names="value")

        self.prev_button.on_click(self._on_prev_clicked)
        self.next_button.on_click(self._on_next_clicked)
        self.play_button.on_click(self._on_play_clicked)
        self.pause_button.on_click(self._on_pause_clicked)

        self.mark_worst_button.on_click(self._on_mark_worst_clicked)
        self.mark_clean_button.on_click(self._on_mark_clean_clicked)
        self.clear_button.on_click(self._on_clear_clicked)

        self.out = widgets.Output()
        self.info_out = widgets.Output()

        self.controls = widgets.VBox([
            self.slider,
            self.segment_selector,
            widgets.HBox([
                self.prev_button,
                self.next_button,
                self.play_button,
                self.pause_button,
            ]),
            self.play_delay,
            self.notes,
            widgets.HBox([
                self.mark_worst_button,
                self.mark_clean_button,
                self.clear_button,
            ]),
        ])

    def _selected_segment_index(self):
        return int(self.segment_selector.value)

    def _jump_to_segment_start(self, segment_index):
        start, _ = self.segment_ranges[segment_index]
        self.slider.value = start

    def _on_segment_change(self, change):
        segment_index = int(change["new"])
        self._jump_to_segment_start(segment_index)

    def _on_mark_worst_clicked(self, _):
        segment_index = self._selected_segment_index()

        self.annotation["worst_congestion_failure_segment_index"] = segment_index
        self.annotation["notes"]["worst_congestion_failure"] = self.notes.value

        self._save_annotation()
        self._render()

    def _on_mark_clean_clicked(self, _):
        segment_index = self._selected_segment_index()

        self.annotation["clearly_clean_segment_index"] = segment_index
        self.annotation["notes"]["clearly_clean_segment"] = self.notes.value

        self._save_annotation()
        self._render()

    def _on_clear_clicked(self, _):
        segment_index = self._selected_segment_index()

        if self.annotation["worst_congestion_failure_segment_index"] == segment_index:
            self.annotation["worst_congestion_failure_segment_index"] = None
            self.annotation["notes"]["worst_congestion_failure"] = ""

        if self.annotation["clearly_clean_segment_index"] == segment_index:
            self.annotation["clearly_clean_segment_index"] = None
            self.annotation["notes"]["clearly_clean_segment"] = ""

        self._save_annotation()
        self._render()

    def _on_play_clicked(self, _):
        self.is_playing = True

        while self.is_playing and self.current_timestep < self.num_steps - 1:
            self.slider.value = self.current_timestep + 1
            time.sleep(self.play_delay.value)

        self.is_playing = False

    def _on_pause_clicked(self, _):
        self.is_playing = False

    def _render(self):
        with self.out:
            clear_output(wait=True)

            fig, ax = plt.subplots(figsize=(7, 7))
            ax.set_facecolor("#f7f7f7")

            current_segment = self._segment_for_timestep(self.current_timestep)
            self.segment_selector.value = current_segment

            for r in range(self.num_rows):
                for c in range(self.num_cols):
                    color = "#2f2f2f" if self.obstacles[r, c] else "#ffffff"

                    rect = patches.Rectangle(
                        (c, r),
                        1,
                        1,
                        facecolor=color,
                        edgecolor="#d0d0d0",
                        linewidth=0.5,
                    )
                    ax.add_patch(rect)

            for i, (gr, gc) in enumerate(self.goals):
                color = self.agent_colors[i % len(self.agent_colors)]

                goal = patches.Rectangle(
                    (gc + 0.18, gr + 0.18),
                    0.64,
                    0.64,
                    facecolor="none",
                    edgecolor=color,
                    linewidth=2.5,
                    linestyle="--",
                )
                ax.add_patch(goal)

                ax.text(
                    gc + 0.5,
                    gr + 0.5,
                    str(i),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=color,
                    weight="bold",
                )

            positions_t = self.positions[self.current_timestep]

            for i, (r, c) in enumerate(positions_t):
                color = self.agent_colors[i % len(self.agent_colors)]

                circle = patches.Circle(
                    (c + 0.5, r + 0.5),
                    0.33,
                    facecolor=color,
                    edgecolor="black",
                    linewidth=1.2,
                    alpha=0.95,
                )
                ax.add_patch(circle)

                ax.text(
                    c + 0.5,
                    r + 0.5,
                    str(i),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="white",
                    weight="bold",
                )

            segment_label = f"segment {current_segment}/{self.num_segments - 1}"
            title = (
                f"{self.npz_path.name} | "
                f"timestep {self.current_timestep}/{self.num_steps - 1} | "
                f"{segment_label}"
            )

            if self.annotation["worst_congestion_failure_segment_index"] == current_segment:
                title += " | WORST CONGESTION"

            if self.annotation["clearly_clean_segment_index"] == current_segment:
                title += " | CLEAN"

            ax.set_title(title, fontsize=12, weight="bold", pad=12)

            ax.set_xlim(0, self.num_cols)
            ax.set_ylim(self.num_rows, 0)
            ax.set_aspect("equal")

            ax.set_xticks(np.arange(0, self.num_cols + 1, 1))
            ax.set_yticks(np.arange(0, self.num_rows + 1, 1))
            ax.set_xticklabels([])
            ax.set_yticklabels([])
            ax.tick_params(length=0)

            for spine in ax.spines.values():
                spine.set_visible(False)

            plt.tight_layout()
            plt.show()

        self._render_info()

    def _render_info(self):
        with self.info_out:
            clear_output(wait=True)

            current_segment = self._segment_for_timestep(self.current_timestep)
            start, end = self.segment_ranges[current_segment]

            print(f"Scenario: {self.scenario_id}")
            print(f"Current timestep: {self.current_timestep}")
            print(f"Current segment index: {current_segment}")
            print(f"Current segment timestep range: {start}–{end}")

            if self.segment_diffs is not None:
                print(f"Current segment diff: {int(self.segment_diffs[current_segment])}")

            print("\nEpisode-level labels:")
            print(
                "Worst congestion failure segment:",
                self.annotation["worst_congestion_failure_segment_index"],
            )
            print(
                "Clearly clean segment:",
                self.annotation["clearly_clean_segment_index"],
            )

            print("\nNotes:")
            print(
                "Worst congestion:",
                self.annotation["notes"]["worst_congestion_failure"],
            )
            print(
                "Clean segment:",
                self.annotation["notes"]["clearly_clean_segment"],
            )

            print(f"\nSaved to: {self.save_path}")

    def _on_slider_change(self, change):
        self.current_timestep = change["new"]
        self._render()

    def _on_prev_clicked(self, _):
        self.is_playing = False

        if self.current_timestep > 0:
            self.slider.value = self.current_timestep - 1

    def _on_next_clicked(self, _):
        self.is_playing = False

        if self.current_timestep < self.num_steps - 1:
            self.slider.value = self.current_timestep + 1

    def show(self):
        display(self.controls)
        display(self.out)
        display(self.info_out)
        self._render()

In [13]:
import argparse
import numpy as np
from pathlib import Path


def inspect_segments(npz_path):
    npz_path = Path(npz_path)
    data = np.load(npz_path, allow_pickle=True)

    positions = data["positions"]
    segment_diffs = data["segment_diffs"]

    num_steps = positions.shape[0]
    num_segments = len(segment_diffs)

    # Same assumption as your viewer: evenly split timesteps across segments
    boundaries = np.linspace(0, num_steps, num_segments + 1).astype(int)

    print(f"File: {npz_path}")
    print(f"Num steps: {num_steps}")
    print(f"Num segments: {num_segments}")
    print()

    for i, diff in enumerate(segment_diffs):
        start = int(boundaries[i])
        end = int(boundaries[i + 1] - 1)

        print(f"Segment {i}: timesteps {start}–{end}, segment_diff={int(diff)}")




In [29]:
path = "dataset/held_out/filtered_npzs/maze_ms131_ss1000_na48.npz"
inspect_segments(path)
viewer = NPZReplayViewer(path)
viewer.show()

File: dataset/held_out/filtered_npzs/maze_ms131_ss1000_na48.npz
Num steps: 206
Num segments: 12

Segment 0: timesteps 0–16, segment_diff=13
Segment 1: timesteps 17–33, segment_diff=15
Segment 2: timesteps 34–50, segment_diff=33
Segment 3: timesteps 51–67, segment_diff=0
Segment 4: timesteps 68–84, segment_diff=9
Segment 5: timesteps 85–102, segment_diff=19
Segment 6: timesteps 103–119, segment_diff=21
Segment 7: timesteps 120–136, segment_diff=18
Segment 8: timesteps 137–153, segment_diff=4
Segment 9: timesteps 154–170, segment_diff=20
Segment 10: timesteps 171–187, segment_diff=2
Segment 11: timesteps 188–205, segment_diff=-2


Output()

Output()